# MATH840 — Lab 2, worked example

**One pass through the whole assignment, start to finish, on a series nobody was assigned.**

Queensland clothing retailing, monthly, 1982–2018. It is drawn from `aus_retail`, which is
**not** in the pool your own series comes from — so this is a demonstration of the method,
not a template you can copy answers out of.

The section headings are the ones your submission must have. Watch what goes in each, and
in particular how much of the work is *sentences* rather than code.

## 0. Setup

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from coreforecast.scalers import boxcox, boxcox_lambda

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

SEASON = 12
FREQ = "MS"

retail = pd.read_csv(f"{BASE}/aus_retail.csv", parse_dates=["Month"])
raw = (retail[(retail["State"] == "Queensland")
              & (retail["Industry"] == "Clothing retailing")]
       .sort_values("Month")[["Month", "Turnover"]]
       .rename(columns={"Month": "ds", "Turnover": "y"})
       .reset_index(drop=True))

raw.head()

## 1. Data preparation

*1 point.* Four facts and any repairs the calendar needs.

In [ ]:
print("rows:      ", len(raw))
print("span:      ", raw["ds"].min().date(), "→", raw["ds"].max().date())
print("missing y: ", raw["y"].isna().sum())
print("duplicates:", raw["ds"].duplicated().sum())
print("zeros:     ", int((raw["y"] == 0).sum()))
print("\nobserved spacings:")
print(raw["ds"].diff().value_counts().head())

In [ ]:
grid = pd.date_range(raw["ds"].min(), raw["ds"].max(), freq=FREQ)
s = raw.set_index("ds").reindex(grid).rename_axis("ds").reset_index()

print(f"{len(raw)} rows → {len(s)} on a regular grid; revealed gaps: {int(s['y'].isna().sum())}")
print(f"complete seasonal cycles: {len(s) / SEASON:.1f}")

**Description.** Monthly turnover, April 1982 to December 2018, 441 observations — a little
over 36 complete yearly cycles, which is plenty for seasonal methods. No duplicated
timestamps, no zeros, and the calendar is already regular: reindexing onto a monthly grid
changes nothing and reveals no gaps. Nothing to repair, and saying so *is* the answer to
this section.

## 2. EDA and visualisation

*4 points.* Four graphics, a variance decision, two decompositions — each with a reading.

### 2.1 Graphics

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(s["ds"], s["y"], linewidth=1, color="#e64173")
ax.set_title("Queensland clothing retailing, monthly turnover (AUD million)")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
d = s.dropna(subset=["y"]).copy()
d["year"], d["month"] = d["ds"].dt.year, d["ds"].dt.month

fig, ax = plt.subplots(figsize=(12, 4))
for yr, grp in d.groupby("year"):
    ax.plot(grp["month"], grp["y"], linewidth=0.8, alpha=0.6)
ax.set_xticks(range(1, 13))
ax.set_xlabel("month")
ax.set_title("Seasonal plot: one line per year")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 12, figsize=(13, 3.2), sharey=True)
for (month, grp), ax in zip(d.groupby("month"), axes):
    ax.plot(grp["year"], grp["y"], linewidth=0.9, color="#20B2AA")
    ax.axhline(grp["y"].mean(), color="#e64173", linewidth=1)
    ax.set_title(month, fontsize=8)
    ax.set_xticks([])
axes[0].set_ylabel("turnover")
fig.suptitle("Seasonal subseries: each month over time, with its mean", y=1.04)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
plot_acf(d["y"], lags=48, ax=ax)
ax.set_title("ACF, 48 lags")
plt.show()

**What the four plots say.** A strong upward trend that flattens after about 2010. A very
regular annual cycle: every year peaks in December and bottoms out in February — the
seasonal plot shows the lines stacked but identical in shape. The subseries plot shows the
December panel rising far more steeply than the February panel, which is the first sign
that the seasonal swing is *proportional* to the level rather than fixed in size. The ACF
decays slowly (trend) with clear spikes at lags 12, 24, 36 (seasonality); nothing here is
close to white noise.

### 2.2 Variance

The seasonal swing in the time plot is visibly wider on the right than on the left. That
is exactly the situation an additive decomposition handles badly, so estimate a Box-Cox
$\lambda$ and look at what it fixes.

In [ ]:
y = s["y"].to_numpy(float)
lam = boxcox_lambda(y, method="loglik", season_length=SEASON)
work = boxcox(y, lam)

first, last = y[:60], y[-60:]
first_w, last_w = work[:60], work[-60:]
print(f"lambda = {lam:.3f}")
print(f"  raw:    sd of first 5 years = {first.std():7.2f}, last 5 years = {last.std():7.2f}"
      f"  → ×{last.std() / first.std():.1f}")
print(f"  boxcox: sd of first 5 years = {first_w.std():7.2f}, last 5 years = {last_w.std():7.2f}"
      f"  → ×{last_w.std() / first_w.std():.1f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(s["ds"], y, linewidth=0.9, color="#e64173"); axes[0].set_title("original")
axes[1].plot(s["ds"], work, linewidth=0.9, color="#20B2AA")
axes[1].set_title(f"Box-Cox transformed, $\\lambda$ = {lam:.2f}")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Decision: work transformed.** The spread of the last five years is 6.5 times the spread
of the first five in the original series; after the transformation it is 2.4 times. That is
a large improvement and it makes the additive decompositions below defensible.

Note what the number also says: $\lambda = 0.45$ is roughly a square root, and it does
**not** fully stabilise the variance — a residual growth of 2.4× remains. So this is a
compromise, not a fix, and the remainder in the decomposition below will still be a little
wider on the right. Saying that out loud is worth more than pretending the problem is gone.

### 2.3 Decomposition

Classical and STL, on the transformed series.

In [ ]:
classical = seasonal_decompose(work, period=SEASON, model="additive")

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, (name, comp) in zip(axes, [("observed", classical.observed),
                                   ("trend", classical.trend),
                                   ("seasonal", classical.seasonal),
                                   ("remainder", classical.resid)]):
    ax.plot(s["ds"], comp, linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("Classical decomposition")
plt.tight_layout()
plt.show()

In [ ]:
stl = STL(work, period=SEASON, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, (name, comp) in zip(axes, [("observed", work),
                                   ("trend", stl.trend),
                                   ("seasonal", stl.seasonal),
                                   ("remainder", stl.resid)]):
    ax.plot(s["ds"], comp, linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("STL decomposition (seasonal=13, robust)")
plt.tight_layout()
plt.show()

In [ ]:
season = pd.Series(stl.seasonal, index=s["ds"])
by_month = season.groupby(season.index.month).mean().round(2)
print("average seasonal effect by month (transformed units):")
print(by_month.to_string())

early, late = season.iloc[:120], season.iloc[-120:]
print(f"\nseasonal amplitude: first 10 years = {early.max() - early.min():.2f}, "
      f"last 10 years = {late.max() - late.min():.2f}")

resid = pd.Series(stl.resid, index=s["ds"])
print("largest remainders:", [str(x.date()) for x in resid.abs().nlargest(4).index])

**Reading the components.**

The trend rises steeply from 1982 to about 2008 and then flattens; the last decade is
essentially a plateau, which matters for any method that extrapolates a slope.

The seasonal component is dominated by December (+3.7) against a February trough (−2.3) —
the Christmas trade, worth roughly six units of swing out of a series whose transformed
level is around 25. It is stable in *shape* but not in *size*: the amplitude grows from
4.3 in the first decade to 9.4 in the last, so even after the Box-Cox transform the
seasonality is still becoming more pronounced.

The remainder is small and mostly structureless, with the largest excursions in 1999 and
2007 — worth a look at the calendar before assuming they are noise.

**Classical versus STL.** They agree on the trend everywhere except the ends, where the
classical version is missing six months at each edge — the price of a centred moving
average. They disagree more interestingly on the seasonal component: the classical
decomposition forces one fixed seasonal pattern for all 36 years, so the growing amplitude
we just measured is pushed into the remainder. STL lets the pattern evolve and puts it
where it belongs. On this series **STL is the one to trust**, and the reason is visible in
its remainder rather than a matter of preference.

## 3. Implementation

*2 points.*

### 3.1 Strength of trend and seasonality

In [ ]:
def strengths(fit) -> tuple[float, float]:
    """F_T and F_S from a fitted STL decomposition."""
    R, T, S = fit.resid, fit.trend, fit.seasonal
    return (max(0.0, 1 - R.var() / (T + R).var()),
            max(0.0, 1 - R.var() / (S + R).var()))


F_T, F_S = strengths(stl)
print(f"F_T = {F_T:.2f}   F_S = {F_S:.2f}")

Both are close to 1: the remainder is tiny next to either component. This series is almost
entirely trend plus season, with very little left over.

### 3.2 Seasonally adjusted series

In [ ]:
adjusted = work - stl.seasonal

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(s["ds"], work, linewidth=0.8, alpha=0.45, label="transformed", color="#e64173")
ax.plot(s["ds"], adjusted, linewidth=1.2, label="seasonally adjusted", color="#20B2AA")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Seasonally adjusted series")
plt.show()

**What it is for, and what it hides.** With December and February pulled out, the turning
points are readable: the slowdown after 2008 is obvious here and easy to miss in the raw
series, where every December spike interrupts the eye. What it hides is that the removed
component is the largest single feature of this business — a retailer planning stock cannot
use the adjusted series, because December *is* the year.

### 3.3 The prediction

Which of the four simple methods will be hardest to beat?

In [ ]:
PREDICTION = "snaive"

**The argument.** $F_S = 0.93$ says the seasonal component carries almost all the variance
that is not trend, and the seasonal plot shows the pattern repeating in the same shape for
36 years. Seasonal naive copies exactly that component forward, so it starts with nearly
everything this series does for free.

The competition has no such advantage. `mean` ignores both the trend and the seasonality of
a series that is mostly trend and seasonality. `naive` carries the level of the last month
forward — and since the last month is December, the peak of the year, every forecast it
makes is biased high. `drift` extrapolates the slope from the first observation to the
last, which on a series whose trend flattened after 2008 means projecting a growth rate the
business stopped delivering a decade ago.

So: **snaive**, and the gap should be large rather than marginal.

## 4. Code quality and reproducibility

Restart, run all, check that nothing errors and that every number quoted above still comes
out of a cell. That is the whole point.

## What next week does with this

Your own submission stops at the prediction. For this demonstration, here is the check —
so you know precisely what you are committing to.

In [ ]:
def benchmark_forecasts(y, h, m):
    T = len(y)
    return {
        "mean":   np.repeat(y.mean(), h),
        "naive":  np.repeat(y[-1], h),
        "snaive": np.array([y[-m + (i % m)] for i in range(h)]),
        "drift":  y[-1] + np.arange(1, h + 1) * (y[-1] - y[0]) / (T - 1),
    }


train, test = y[:-12], y[-12:]
rmse = {k: float(np.sqrt(np.mean((test - v) ** 2)))
        for k, v in benchmark_forecasts(train, 12, SEASON).items()}

for name, value in sorted(rmse.items(), key=lambda kv: kv[1]):
    mark = "  ← predicted" if name == PREDICTION else ""
    print(f"  {name:7s} RMSE {value:7.1f}{mark}")

Seasonal naive is seven times more accurate than anything else on the shelf. That is the
bar the rest of the course is spent trying to clear — and the reason the argument above
mattered more than the code.